# Sequence to Sequence Learning with Neural Networks

# WMT14 English-French reproduction on a single Colab T4.


## 1. Setup


In [ ]:
!pip install -q sacrebleu sacremoses datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 2.4 MB/s eta 0:00:00


In [ ]:
import math, random, time, re
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sacremoses import MosesTokenizer, MosesDetokenizer
import sacrebleu

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 2. Config

Main settings for the run.


In [ ]:
# Data
NUM_TRAIN_SENTENCES = 200_000
NUM_VAL_SENTENCES = 1_000
MAX_LEN = 30

# Vocabulary
SRC_VOCAB_SIZE = 40_000
TGT_VOCAB_SIZE = 40_000

# Model (paper uses EMB_DIM = HID_DIM = 1000; halved here so it trains
# comfortably for many epochs on a single T4)
EMB_DIM = 500
HID_DIM = 500
NUM_LAYERS = 4
DROPOUT = 0.2  # a little regularization so 60-100 passes over 200k pairs don't just memorize

# Training
BATCH_SIZE = 128
INIT_LR = 0.5          # static SGD learning rate -- no decay schedule, held fixed the whole run
EPOCHS = 200            # real, full passes over the 200k training pairs (in the 60-100 range you asked for)
GRAD_CLIP_NORM = 5.0
REVERSE_SOURCE = True
NUM_WORKERS = 2
PIN_MEMORY = True

# Decoding
BEAM_SIZE = 5
MAX_DECODE_LEN = 40


## 3. Data

Using WMT14 English-French. I stream the dataset, keep sentence pairs up to 30 tokens, and take 200k training pairs plus 1000 validation pairs.


In [ ]:
from datasets import load_dataset

mt_en = MosesTokenizer(lang="en")
mt_fr = MosesTokenizer(lang="fr")

def tokenize(text, tokenizer):
    return tokenizer.tokenize(text, escape=False)

_data_start = time.time()
raw = load_dataset("wmt/wmt14", "fr-en", split="train", streaming=True)

train_pairs = []
val_pairs = []
seen = 0
PROGRESS_EVERY = 50_000
for ex in raw:
    src = ex["translation"]["en"]
    tgt = ex["translation"]["fr"]
    src_toks = tokenize(src, mt_en)
    tgt_toks = tokenize(tgt, mt_fr)
    if 1 <= len(src_toks) <= MAX_LEN and 1 <= len(tgt_toks) <= MAX_LEN:
        if len(train_pairs) < NUM_TRAIN_SENTENCES:
            train_pairs.append((src_toks, tgt_toks))
        elif len(val_pairs) < NUM_VAL_SENTENCES:
            val_pairs.append((src_toks, tgt_toks))
        else:
            break
    seen += 1
    if seen % PROGRESS_EVERY == 0:
        print(f"  scanned {seen:>9,} raw examples | collected {len(train_pairs):>9,}/{NUM_TRAIN_SENTENCES:,} train "
              f"+ {len(val_pairs)}/{NUM_VAL_SENTENCES} val")

print(f"Collected {len(train_pairs):,} train + {len(val_pairs):,} val pairs "
      f"(scanned {seen:,} raw examples to fill the length filter) "
      f"in {(time.time() - _data_start)/60:.1f} min.")


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

  scanned    50,000 raw examples | collected    27,854/200,000 train + 0/1000 val
  scanned   100,000 raw examples | collected    55,400/200,000 train + 0/1000 val
  scanned   150,000 raw examples | collected    83,449/200,000 train + 0/1000 val
  scanned   200,000 raw examples | collected   111,590/200,000 train + 0/1000 val
  scanned   250,000 raw examples | collected   138,839/200,000 train + 0/1000 val
  scanned   300,000 raw examples | collected   165,523/200,000 train + 0/1000 val
  scanned   350,000 raw examples | collected   193,889/200,000 train + 0/1000 val
Collected 200,000 train + 1,000 val pairs (scanned 362,665 raw examples to fill the length filter) in 2.7 min.


In [ ]:
# Build vocabularies ---
PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
SPECIALS = [PAD, SOS, EOS, UNK]

def build_vocab(pairs, idx, max_size):
    counter = Counter()
    for pair in pairs:
        counter.update(pair[idx])
    most_common = [w for w, _ in counter.most_common(max_size - len(SPECIALS))]
    itos = SPECIALS + most_common
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

src_stoi, src_itos = build_vocab(train_pairs, 0, SRC_VOCAB_SIZE)
tgt_stoi, tgt_itos = build_vocab(train_pairs, 1, TGT_VOCAB_SIZE)


def encode(tokens, stoi, add_eos=True, reverse=False):
    ids = [stoi.get(t, stoi[UNK]) for t in tokens]
    if reverse:
        ids = ids[::-1]
    if add_eos:
        ids = ids + [stoi[EOS]]
    return ids


In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_stoi, tgt_stoi, reverse_source=True):
        self.src_data = []
        self.tgt_data = []
        for src_toks, tgt_toks in pairs:
            src_ids = encode(src_toks, src_stoi, add_eos=True, reverse=reverse_source)
            tgt_ids = [tgt_stoi[SOS]] + encode(tgt_toks, tgt_stoi, add_eos=True)
            self.src_data.append(torch.tensor(src_ids, dtype=torch.long))
            self.tgt_data.append(torch.tensor(tgt_ids, dtype=torch.long))

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, i):
        return self.src_data[i], self.tgt_data[i]


def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_lens = torch.tensor([len(s) for s in srcs], dtype=torch.long)
    tgt_lens = torch.tensor([len(t) for t in tgts], dtype=torch.long)
    src_pad = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=src_stoi[PAD])
    tgt_pad = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=tgt_stoi[PAD])
    return src_pad, src_lens, tgt_pad, tgt_lens

train_ds = TranslationDataset(train_pairs, src_stoi, tgt_stoi, reverse_source=REVERSE_SOURCE)
val_ds = TranslationDataset(val_pairs, src_stoi, tgt_stoi, reverse_source=REVERSE_SOURCE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

STEPS_PER_EPOCH = len(train_loader)  # one real epoch = one full pass over train_ds
print(f"train pairs: {len(train_ds):,} | val pairs: {len(val_ds):,} | steps/epoch: {STEPS_PER_EPOCH:,}")


train pairs: 200,000 | val pairs: 1,000 | steps/epoch: 1,563


## 4. Model

Separate encoder and decoder LSTMs with no attention.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, pad_idx, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)

    def forward(self, src, src_lens):
        emb = self.embedding(src)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, src_lens.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h, c) = self.lstm(packed)
        return h, c  # each: (num_layers, batch, hid_dim) -- the fixed-length sentence representation


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, pad_idx, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hid_dim, vocab_size)

    def forward(self, tgt_in, h, c):
        emb = self.embedding(tgt_in)
        out, (h, c) = self.lstm(emb, (h, c))
        logits = self.fc_out(out)
        return logits, h, c


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_lens, tgt_in):
        h, c = self.encoder(src, src_lens)
        logits, _, _ = self.decoder(tgt_in, h, c)
        return logits


encoder = Encoder(len(src_itos), EMB_DIM, HID_DIM, NUM_LAYERS, src_stoi[PAD], DROPOUT)
decoder = Decoder(len(tgt_itos), EMB_DIM, HID_DIM, NUM_LAYERS, tgt_stoi[PAD], DROPOUT)
model = Seq2Seq(encoder, decoder).to(device)

n_params = sum(p.numel() for p in model.parameters())


## 5. Training

SGD with a fixed (static) learning rate for the whole run -- no decay schedule.


### Objective

The training objective follows the paper: maximize the average log probability of the correct target sequence given the source sequence. The code minimizes the negative token log-likelihood with padding excluded.


In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/seq2seq_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, "seq2seq_200k_t4_paper_method.pt")


Mounted at /content/drive


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=INIT_LR, momentum=0.0)

TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
BEST_CKPT_PATH = os.path.join(CKPT_DIR, "seq2seq_200k_t4_paper_method_best.pt")
SAVE_EVERY_STEPS = 500  # also checkpoint mid-epoch, not just at epoch end

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    global_step = ckpt["global_step"]
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    resume_epoch = global_step // STEPS_PER_EPOCH + 1
    step_in_epoch = global_step % STEPS_PER_EPOCH
    print(f"Resuming from checkpoint: epoch {resume_epoch}/{EPOCHS} "
          f"({step_in_epoch}/{STEPS_PER_EPOCH} steps into it), "
          f"global_step={global_step:,}, best_val_loss so far={best_val_loss:.3f}")
else:
    global_step = 0
    best_val_loss = float("inf")
    print("No checkpoint found -- starting training from scratch.")

# Learning rate is intentionally static: every param group just keeps INIT_LR
# for the entire run (no warmup, no decay, no schedule).


def save_checkpoint():
    torch.save(
        {
            "global_step": global_step,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
        },
        CKPT_PATH,
    )


def negative_log_likelihood(logits, targets):
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = torch.gather(
        log_probs,
        -1,
        targets.unsqueeze(-1)
    ).squeeze(-1)
    mask = targets != tgt_stoi[PAD]
    return -(token_log_probs.masked_select(mask).mean())


def run_train_steps(num_steps):
    global global_step
    model.train()
    total_loss = 0.0
    total_tokens = 0
    loader_iter = iter(train_loader)
    step_start = time.time()

    for i in range(num_steps):
        try:
            src, src_lens, tgt, _ = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            src, src_lens, tgt, _ = next(loader_iter)

        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        optimizer.zero_grad(set_to_none=True)
        logits = model(src, src_lens, tgt_in)
        loss = negative_log_likelihood(logits, tgt_out)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        n_tok = (tgt_out != tgt_stoi[PAD]).sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok
        global_step += 1

        if (i + 1) % 100 == 0:
            elapsed = time.time() - step_start
            sec_per_step = elapsed / (i + 1)
            print(f"    step {i+1}/{num_steps} | loss {loss.item():.3f} | {sec_per_step:.2f}s/step")

        if global_step % SAVE_EVERY_STEPS == 0:
            save_checkpoint()

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def validation_loss():
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for src, src_lens, tgt, _ in val_loader:
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        logits = model(src, src_lens, tgt_in)
        loss = negative_log_likelihood(logits, tgt_out)
        n_tok = (tgt_out != tgt_stoi[PAD]).sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok

    return total_loss / max(total_tokens, 1)


history = []
start_time = time.time()

while global_step < TOTAL_STEPS:
    remaining = TOTAL_STEPS - global_step
    steps_this_epoch = min(STEPS_PER_EPOCH, remaining)
    epoch = global_step // STEPS_PER_EPOCH + 1

    print(f"\n=== Epoch {epoch:>3d}/{EPOCHS} starting ({steps_this_epoch} steps) ===")
    epoch_start = time.time()

    train_loss = run_train_steps(steps_this_epoch)
    print(f"Epoch {epoch:>3d}/{EPOCHS} | train loss {train_loss:.3f} "
          f"(took {(time.time() - epoch_start)/60:.1f} min) | running validation...")

    val_loss = validation_loss()
    history.append((global_step, train_loss, val_loss, INIT_LR))

    print(f"Epoch {epoch:>3d}/{EPOCHS} | train {train_loss:.3f} | val {val_loss:.3f} | lr {INIT_LR:.4g}")

    save_checkpoint()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {"model_state": model.state_dict(), "val_loss": val_loss, "epoch": epoch},
            BEST_CKPT_PATH,
        )
        print(f"  -> new best val loss {best_val_loss:.3f}, saved to {BEST_CKPT_PATH}")

print(f"Training finished in {(time.time() - start_time)/60:.1f} min. Best val loss: {best_val_loss:.3f}")


Resuming from checkpoint: epoch 196/200 (215/1563 steps into it), global_step=305,000, best_val_loss so far=2.848

=== Epoch 196/200 starting (1563 steps) ===
    step 100/1563 | loss 1.058 | 0.30s/step
    step 200/1563 | loss 1.066 | 0.31s/step
    step 300/1563 | loss 1.002 | 0.32s/step
    step 400/1563 | loss 1.091 | 0.32s/step
    step 500/1563 | loss 1.132 | 0.32s/step
    step 600/1563 | loss 1.142 | 0.32s/step
    step 700/1563 | loss 1.038 | 0.32s/step
    step 800/1563 | loss 1.106 | 0.32s/step
    step 900/1563 | loss 1.026 | 0.32s/step
    step 1000/1563 | loss 1.127 | 0.32s/step
    step 1100/1563 | loss 1.117 | 0.32s/step
    step 1200/1563 | loss 1.124 | 0.32s/step
    step 1300/1563 | loss 1.013 | 0.33s/step
    step 1400/1563 | loss 1.074 | 0.33s/step
    step 1500/1563 | loss 1.096 | 0.32s/step
Epoch 196/200 | train loss 1.093 (took 8.5 min) | running validation...
Epoch 196/200 | train 1.093 | val 3.631 | lr 0.5

=== Epoch 197/200 starting (1563 steps) ===
    step 

## 6. Decoding

Greedy decoding and beam search (beam size = BEAM_SIZE, best-checkpoint weights) are used to compare the outputs.


In [ ]:
# Use the checkpoint with the lowest validation loss for inference,
# not necessarily the last one trained.
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(best_ckpt["model_state"])
model.eval()
print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']} (val loss {best_ckpt['val_loss']:.3f})")


Loaded best checkpoint from epoch 57 (val loss 2.848)


In [ ]:
@torch.no_grad()
def greedy_decode(model, src_tokens, max_len=MAX_DECODE_LEN):
    model.eval()
    src_ids = encode(src_tokens, src_stoi, add_eos=True, reverse=REVERSE_SOURCE)
    src = torch.tensor([src_ids], device=device)
    src_lens = torch.tensor([len(src_ids)])

    h, c = model.encoder(src, src_lens)
    inp = torch.tensor([[tgt_stoi[SOS]]], device=device)
    out_ids = []

    for _ in range(max_len):
        logits, h, c = model.decoder(inp, h, c)
        next_id = logits[:, -1, :].argmax(-1).item()
        if next_id == tgt_stoi[EOS]:
            break
        out_ids.append(next_id)
        inp = torch.tensor([[next_id]], device=device)

    return [tgt_itos[i] for i in out_ids]


@torch.no_grad()
def beam_search_decode(model, src_tokens, beam_size=BEAM_SIZE, max_len=MAX_DECODE_LEN):
    model.eval()
    src_ids = encode(src_tokens, src_stoi, add_eos=True, reverse=REVERSE_SOURCE)
    src = torch.tensor([src_ids], device=device)
    src_lens = torch.tensor([len(src_ids)])
    h0, c0 = model.encoder(src, src_lens)

    beams = [(0.0, [tgt_stoi[SOS]], h0, c0, False)]
    completed = []

    for _ in range(max_len):
        candidates = []

        for log_prob, ids, h, c, finished in beams:
            if finished:
                candidates.append((log_prob, ids, h, c, True))
                continue

            inp = torch.tensor([[ids[-1]]], device=device)
            logits, h_new, c_new = model.decoder(inp, h, c)
            log_probs = F.log_softmax(logits[:, -1, :], dim=-1).squeeze(0)
            topk_lp, topk_idx = log_probs.topk(beam_size)

            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                done = idx == tgt_stoi[EOS]
                candidates.append((log_prob + lp, ids + [idx], h_new, c_new, done))

        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_size]
        completed.extend([b for b in beams if b[4]])

        if all(b[4] for b in beams):
            break

    best = max(completed if completed else beams, key=lambda x: x[0])
    ids = [i for i in best[1][1:] if i != tgt_stoi[EOS]]
    return [tgt_itos[i] for i in ids]


In [ ]:
mdt_fr = MosesDetokenizer(lang="fr")

for src_toks, tgt_toks in val_pairs[:5]:
    src_text = " ".join(src_toks)
    pred = mdt_fr.detokenize(beam_search_decode(model, src_toks))
    ref = mdt_fr.detokenize(tgt_toks)
    print(f"SRC:  {src_text}")
    print(f"PRED: {pred}")
    print(f"REF:  {ref}")
    print()


SRC:  In fact , this report reveals a clear political colour and the ' values and standards ' tag that goes with it .
PRED: En fait, ce rapport montre clairement un équilibre politique et les valeurs et les sanctions.
REF:  Ce rapport est en effet empreint d'une couleur politique évidente, avec tout l'éventail de "normes et valeurs" assorti à cette couleur.

SRC:  A constitution of our own would be undesirable and unnecessary .
PRED: Une telle situation serait une bonne chose et ne serait pas acceptable.
REF:  L'adoption d'une constitution propre est indésirable et superfétatoire.

SRC:  The EU consists of solid nation states with their own perfectly good constitutions .
PRED: La Communauté de l'UE n'a pas fait ses responsabilités, et c'est une bonne chose.
REF:  L'Union est composée d'États nations solidement établis avec des constitutions qui fonctionnent bien.

SRC:  These are contained in an existing democratic order with a process of judicial review and an involved civilian popula

## 7. BLEU evaluation

Calculate BLEU on the validation set using sacrebleu.


In [ ]:
@torch.no_grad()
def evaluate_bleu(model, pairs, decode_fn=beam_search_decode):
    hyps, refs = [], []
    for src_toks, tgt_toks in pairs:
        hyps.append(mdt_fr.detokenize(decode_fn(model, src_toks)))
        refs.append(mdt_fr.detokenize(tgt_toks))
    return sacrebleu.corpus_bleu(hyps, [refs]).score

bleu = evaluate_bleu(model, val_pairs, beam_search_decode)
print(f"BLEU: {bleu:.2f}")


BLEU: 12.38


## 8. Encoder representation

Use PCA on a few sentence representations from the encoder.


In [ ]:
# Save everything needed to run inference in a separate Colab notebook:
# model weights (best validation checkpoint) + config + vocabularies.
INFERENCE_PKG_PATH = os.path.join(CKPT_DIR, "seq2seq_inference_package.pt")

# Use the best validation checkpoint, not necessarily the last epoch trained.
best_ckpt_for_export = torch.load(BEST_CKPT_PATH, map_location="cpu")

torch.save(
    {
        "model_state": best_ckpt_for_export["model_state"],
        "config": {
            "EMB_DIM": EMB_DIM,
            "HID_DIM": HID_DIM,
            "NUM_LAYERS": NUM_LAYERS,
            "DROPOUT": DROPOUT,
            "MAX_LEN": MAX_LEN,
            "MAX_DECODE_LEN": MAX_DECODE_LEN,
            "BEAM_SIZE": BEAM_SIZE,
            "REVERSE_SOURCE": REVERSE_SOURCE,
        },
        "src_stoi": src_stoi,
        "src_itos": src_itos,
        "tgt_stoi": tgt_stoi,
        "tgt_itos": tgt_itos,
        "specials": {"PAD": PAD, "SOS": SOS, "EOS": EOS, "UNK": UNK},
        "trained_epoch": best_ckpt_for_export.get("epoch"),
        "val_loss": best_ckpt_for_export.get("val_loss"),
    },
    INFERENCE_PKG_PATH,
)

print(f"Saved standalone inference package to: {INFERENCE_PKG_PATH}")
print(f"  (from best checkpoint: epoch {best_ckpt_for_export.get('epoch')}, "
      f"val loss {best_ckpt_for_export.get('val_loss'):.3f})")
print()

Saved standalone inference package to: /content/drive/MyDrive/seq2seq_checkpoints/seq2seq_inference_package.pt
  (from best checkpoint: epoch 57, val loss 2.848)

